# Split Large Images into tiles

## Rationale

Large images (e.g. acquired with the auto-stitching functionality *Large Image* of NIS) can quickly take up several gigabyes and become prohibitively hard to analyze and visualize on older computers. Therefore, this recipe will split the large images and save virtual tiles as TIFF files.

In [ ]:
from itertools import product
from pathlib import Path
import numpy as np
from nd2 import ND2File
from calmutils.imageio.tiff_imagej import save_tiff_imagej

In [ ]:
in_path = Path('/data/agl_data/Antibody_LacO_QualityControl/23AM06-01/')

out_folder_name = 'split_tiles'
out_path = in_path / out_folder_name

# how many tiles to split into
n_tiles_y, n_tiles_x = 3, 3

in_files = sorted(in_path.glob('*.nd2'))
in_files

In [ ]:
for in_file in in_files:

    # read image, pixelsize
    with ND2File(in_file) as fd:
        img = fd.asarray()
        pixel_size = fd.voxel_size()[::-1]

    # split
    tiles = []
    # 1. split into x "columns"
    tiles_xsplit = np.split(img, n_tiles_x, -1)
    # 2. loop over x "columns", split by y "rows" into tiles
    for tile_xsplit in tiles_xsplit:
        tiles.extend(np.split(tile_xsplit, n_tiles_y, -2))

    # make output directory
    if not out_path.exists():
        out_path.mkdir()

    # save tiles
    for i, (yidx, xidx) in enumerate(product(range(n_tiles_y), range(n_tiles_x))):
        filename = in_file.stem + f'_split_y{yidx}_x{xidx}.tif'
        save_tiff_imagej(out_path / filename, tiles[i], axes='zcyx', pixel_size=pixel_size, distance_unit='micron')

    print(f'finished splitting {in_file}.')